In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "PSO"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# ========= 2) 参数模板 =========
params = {
    "dim": 2,                            # 变量维度
    "lb": np.array([-10.0, -10.0]),      # 下界
    "ub": np.array([10.0, 10.0]),        # 上界
    "n_particles": 30,                   # 粒子数
    "n_iter": 100,                       # 迭代次数
    "w": 0.7, "c1": 1.5, "c2": 1.5       # PSO超参数
}

def obj(x): return np.sum((x-2)**2)

X = params["lb"] + (params["ub"] - params["lb"]) * np.random.rand(params["n_particles"], params["dim"])
V = np.zeros_like(X)
pbest = X.copy()
pbest_val = np.array([obj(x) for x in X])
gidx = np.argmin(pbest_val)
gbest, gbest_val = pbest[gidx].copy(), pbest_val[gidx]

for _ in range(params["n_iter"]):
    r1 = np.random.rand(params["n_particles"], params["dim"])
    r2 = np.random.rand(params["n_particles"], params["dim"])
    V = params["w"]*V + params["c1"]*r1*(pbest-X) + params["c2"]*r2*(gbest-X)
    X = np.clip(X + V, params["lb"], params["ub"])
    vals = np.array([obj(x) for x in X])
    mask = vals < pbest_val
    pbest[mask], pbest_val[mask] = X[mask], vals[mask]
    idx = np.argmin(pbest_val)
    if pbest_val[idx] < gbest_val:
        gbest_val, gbest = pbest_val[idx], pbest[idx].copy()

print(gbest, gbest_val)


In [ ]:
"""
粒子群算法

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "粒子群算法.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
DIMENSION = 2  # TODO: 请填写[变量维度]，说明：决策变量个数。
LOWER_BOUND = -5  # TODO: 请填写[变量下界]，说明：可扩展为数组。
UPPER_BOUND = 5  # TODO: 请填写[变量上界]，说明：必须大于下界。
PARTICLE_COUNT = 40  # TODO: 请填写[粒子数量]，说明：越大搜索更充分但更慢。
MAX_ITER = 100  # TODO: 请填写[最大迭代次数]，说明：正整数。
W = 0.7  # TODO: 请填写[惯性权重]，说明：常用 0.4 到 0.9。
C1 = 1.5  # TODO: 请填写[个体学习因子]，说明：正数。
C2 = 1.5  # TODO: 请填写[群体学习因子]，说明：正数。
TARGET_VECTOR = np.array([1.0, 2.0])  # TODO: 请填写[示例目标向量]，说明：请替换为真实目标函数参数。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def objective(x):
    return np.sum((x - TARGET_VECTOR) ** 2)


def run_model(data: pd.DataFrame) -> None:
    rng = np.random.default_rng(RANDOM_STATE)
    pos = rng.uniform(LOWER_BOUND, UPPER_BOUND, size=(PARTICLE_COUNT, DIMENSION))
    vel = np.zeros_like(pos)
    pbest = pos.copy()
    pbest_score = np.array([objective(p) for p in pos])
    gbest = pbest[np.argmin(pbest_score)].copy()
    for _ in range(MAX_ITER):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = W * vel + C1 * r1 * (pbest - pos) + C2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, LOWER_BOUND, UPPER_BOUND)
        scores = np.array([objective(p) for p in pos])
        improved = scores < pbest_score
        pbest[improved], pbest_score[improved] = pos[improved], scores[improved]
        gbest = pbest[np.argmin(pbest_score)].copy()
    print("全局最优:", gbest, "目标值:", objective(gbest))


if __name__ == "__main__":
    df = load_data()
    run_model(df)
